# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravindrathalari06/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("FlyRank/internship-starter")

df = dataset["train"].to_pandas()
print("Rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())

print("\nDate/time-related columns:")
date_cols = [col for col in df.columns if "date" in col.lower() or "time" in col.lower()]
print(date_cols)

Rows: 30000
Unique content IDs: 30000

Date/time-related columns:
['days_since_last_update', 'is_initial_refresh_candidate']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-04 Section 2: verify field buckets

all_fields = df.columns.tolist()

feature_fields = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier"
]

label_fields = []

context_fields = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

excluded_fields = [
    "trend_direction",
    "trend_pct"
]

buckets = {
    "feature": feature_fields,
    "label": label_fields,
    "context": context_fields,
    "excluded": excluded_fields
}

assigned = [field for fields in buckets.values() for field in fields]

print("Total dataset fields:", len(all_fields))
print("Assigned fields:", len(assigned))
print("Unassigned fields:", sorted(set(all_fields) - set(assigned)))

print("\nField counts:")
for bucket, fields in buckets.items():
    print(f"{bucket}: {len(fields)}")

Total dataset fields: 53
Assigned fields: 44
Unassigned fields: ['ai_opportunity', 'health_score', 'is_declining', 'is_initial_refresh_candidate', 'is_quick_win', 'is_underperformer', 'needs_ctr_fix', 'needs_engagement_fix', 'needs_indexing']

Field counts:
feature: 38
label: 0
context: 4
excluded: 2


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-04 Section 3: verify the data contract

# 1. Grain: one row per content item
print("=== GRAIN ===")
print("Total rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("One row per content item:", len(df) == df["content_id"].nunique())

# 2. Counts
print("\n=== COUNTS ===")
print("Total content items:", df["content_id"].nunique())
print("Total columns:", len(df.columns))

# 3. Missing values in planned fields
print("\n=== MISSING VALUES ===")
planned_fields = feature_fields + label_fields + context_fields + excluded_fields

missing = df[planned_fields].isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("No missing values in planned fields.")
else:
    print(missing)
#  time/window check

print("\n=== TIME / WINDOW CHECK ===")

# Check for actual calendar-date/timestamp columns by data type
date_dtype_columns = df.select_dtypes(
    include=["datetime64[ns]", "datetime64[ns, UTC]"]
).columns.tolist()

print("Explicit calendar-date columns:", date_dtype_columns)

print(
    "days_since_last_update range:",
    df["days_since_last_update"].min(),
    "to",
    df["days_since_last_update"].max(),
    "days"
)

print("Calendar-date window available:", len(date_dtype_columns) > 0)

=== GRAIN ===
Total rows: 30000
Unique content IDs: 30000
One row per content item: True

=== COUNTS ===
Total content items: 30000
Total columns: 53

=== MISSING VALUES ===
provider_used        21438
char_count            7699
word_count_tier       7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64

=== TIME / WINDOW CHECK ===
Explicit calendar-date columns: []
days_since_last_update range: 1 to 373 days
Calendar-date window available: False


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-04 Section 4: verify data limits

print("=== DATA LIMITS ===")

# 1. Uneven history / content age
print("\nContent age range:")
print(
    "content_age_days:",
    df["content_age_days"].min(),
    "to",
    df["content_age_days"].max()
)

print("\nDays with impressions range:")
print(
    "days_with_impressions:",
    df["days_with_impressions"].min(),
    "to",
    df["days_with_impressions"].max()
)

# 2. Missing performance history
print("\nMissing performance fields:")
performance_fields = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d"
]

for col in performance_fields:
    print(f"{col}: {df[col].isna().sum():,} missing")

# 3. Calendar-window limitation
date_columns = df.select_dtypes(
    include=["datetime64[ns]", "datetime64[ns, UTC]"]
).columns.tolist()

print("\nExplicit calendar-date columns:", date_columns)
print("Calendar-date window available:", len(date_columns) > 0)

# 4. Relative update-age information
print(
    "\ndays_since_last_update range:",
    df["days_since_last_update"].min(),
    "to",
    df["days_since_last_update"].max(),
    "days"
)

=== DATA LIMITS ===

Content age range:
content_age_days: 90 to 564

Days with impressions range:
days_with_impressions: 1 to 88

Missing performance fields:
impressions_90d: 0 missing
clicks_90d: 0 missing
pageviews_90d: 0 missing
sessions_90d: 0 missing

Explicit calendar-date columns: []
Calendar-date window available: False

days_since_last_update range: 1 to 373 days


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.